In [ ]:
# Core dependencies
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# MMDetection ecosystem
!pip install openmim
!mim install mmengine
!mim install mmcv
!mim install mmdet
!mim install mmpose  # optional if you want advanced transforms

# Image augmentation
!pip install albumentations opencv-python scikit-image tqdm

In [ ]:
!pip install cython
!pip install 'git+https://github.com/facebookresearch/fvcore'
!pip install 'git+https://github.com/cocodataset/cocoapi.git#subdirectory=PythonAPI'
!pip install opencv-python

In [ ]:
!python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

In [ ]:
# Downgrade numpy to avoid compatibility issues
!pip install numpy==1.25.2

# Install PyTorch (GPU, CUDA 11.8)
!pip install torch==2.1.2+cu118 torchvision==0.16.2+cu118 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu118

# Install Detectron2 prebuilt wheels (Windows + CUDA 11.8 + PyTorch 2.1)
!pip install https://dl.fbaipublicfiles.com/detectron2/wheels/cu118/torch2.1/detectron2-1.2.1%2Bcu118-cp39-cp39-win_amd64.whl

# Install Mask2Former dependencies
!pip install opencv-python albumentations scipy scikit-image tqdm

In [ ]:
!pip install numpy==1.25.2

In [ ]:
pip uninstall opencv-python opencv-python-headless -y

In [ ]:
pip install opencv-python==4.9.0.80 opencv-python-headless==4.9.0.80

In [ ]:
import torch

print("GPU Available:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())
print("GPU Name:", torch.cuda.get_device_name(0))

In [ ]:
import numpy as np
import cv2

print("NumPy version:", np.__version__)
print("OpenCV version:", cv2.__version__)

In [ ]:
# ===============================
# 1️⃣ Setup Environment & Imports
# ===============================
import os
import json
import cv2
import torch
import random
import numpy as np
from tqdm import tqdm
import albumentations as A

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch import nn, optim

from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from scipy import ndimage as ndi

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

In [ ]:
# ===============================
# 2️⃣ Set Paths
# ===============================
BASE_DIR = r"E:\Prithu\Sustain\sustain train"
OUTPUT_DIR = os.path.join(BASE_DIR, "m2f_hybrid_dataset")
os.makedirs(os.path.join(OUTPUT_DIR, "train/images"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "val/images"), exist_ok=True)

In [ ]:
import albumentations as A
from tqdm import tqdm
import json, os, cv2

# ===============================
# 3️⃣ Data Preparation + Augmentation
# ===============================
with open(os.path.join(BASE_DIR, "sustain_fixed.json")) as f:
    data = json.load(f)

import albumentations as A

aug = A.Compose([
    A.RandomResizedCrop(size=(1024, 1024), scale=(0.8, 1.0), p=1.0),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.CLAHE(p=0.3),
    A.GaussNoise(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.ElasticTransform(alpha=1, sigma=50, p=0.3),
], bbox_params=A.BboxParams(
    format='coco',
    label_fields=['category_ids'],
    min_area=1,
    check_each_transform=True
))

def prepare_split(split_name, num_per_img=5):
    new_imgs, new_anns = [], []
    ann_id = 1
    for img_info in tqdm(data['images'], desc=f"Processing {split_name}"):
        img_path = os.path.join(BASE_DIR, "images", img_info['file_name'])
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        anns = [a for a in data['annotations'] if a['image_id'] == img_info['id']]

        # Normalize bounding boxes to 0–1 range for albumentations
        norm_boxes = [[x / img.shape[1], y / img.shape[0], w / img.shape[1], h / img.shape[0]] for x, y, w, h in [a['bbox'] for a in anns]]

        for i in range(num_per_img):
            transformed = aug(image=img,
                              bboxes=norm_boxes,
                              category_ids=[a['category_id'] for a in anns])

            # Convert bboxes back to absolute pixel coordinates
            boxes_px = [[bx * img.shape[1], by * img.shape[0], bw * img.shape[1], bh * img.shape[0]] for bx, by, bw, bh in transformed['bboxes']]

            fname = f"{split_name}_{img_info['id']}_{i}.jpg"
            cv2.imwrite(os.path.join(OUTPUT_DIR, f"{split_name}/images", fname),
                        cv2.cvtColor(transformed['image'], cv2.COLOR_RGB2BGR))

            img_id = len(new_imgs) + 1
            new_imgs.append({"id": img_id, "file_name": fname, "width": 1024, "height": 1024})

            for box, cat in zip(boxes_px, transformed['category_ids']):
                x, y, w, h = box
                poly = [x, y, x+w, y, x+w, y+h, x, y+h]
                new_anns.append({
                    "id": ann_id,
                    "image_id": img_id,
                    "category_id": cat,
                    "bbox": box,
                    "area": w*h,
                    "segmentation": [poly],
                    "iscrowd": 0
                })
                ann_id += 1

    with open(os.path.join(OUTPUT_DIR, f"{split_name}.json"), "w") as f:
        json.dump({"images": new_imgs, "annotations": new_anns, "categories": data['categories']}, f)

# Test with small num_per_img first
prepare_split("train", num_per_img=5)
prepare_split("val", num_per_img=2)

In [ ]:
# ===============================
# 4️⃣ Custom Dataset Class for PyTorch
# ===============================
class ParticleDataset(Dataset):
    def __init__(self, images_dir, ann_file, transform=None):
        with open(ann_file) as f:
            data = json.load(f)
        self.images_dir = images_dir
        self.transform = transform
        self.imgs = {img['id']: img for img in data['images']}
        self.anns = data['annotations']

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_info = list(self.imgs.values())[idx]
        img_path = os.path.join(self.images_dir, img_info['file_name'])
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Collect masks for this image
        masks = []
        for ann in [a for a in self.anns if a['image_id']==img_info['id']]:
            x, y, w, h = map(int, ann['bbox'])
            mask = np.zeros((img_info['height'], img_info['width']), dtype=np.uint8)
            mask[y:y+h, x:x+w] = 1
            masks.append(mask)
        mask = np.stack(masks).max(axis=0) if masks else np.zeros((img_info['height'], img_info['width']))

        if self.transform:
            img = self.transform(img)

        img_tensor = torch.tensor(img.transpose(2,0,1), dtype=torch.float32)/255.0
        mask_tensor = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)

        return img_tensor, mask_tensor

In [ ]:
# ===============================
# 5️⃣ Dataloader
# ===============================
train_dataset = ParticleDataset(
    os.path.join(OUTPUT_DIR, "train/images"),
    os.path.join(OUTPUT_DIR, "train.json"),
    transform=None
)

val_dataset = ParticleDataset(
    os.path.join(OUTPUT_DIR, "val/images"),
    os.path.join(OUTPUT_DIR, "val.json"),
    transform=None
)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1)

In [ ]:
# ===============================
# 6️⃣ Simple U-Net Architecture
# ===============================
class UNetBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1):
        super().__init__()
        self.enc1 = UNetBlock(in_ch, 64)
        self.enc2 = UNetBlock(64, 128)
        self.enc3 = UNetBlock(128, 256)
        self.pool = nn.MaxPool2d(2)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = UNetBlock(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = UNetBlock(128, 64)
        self.final = nn.Conv2d(64, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        d2 = self.up2(e3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return torch.sigmoid(self.final(d1))

device = "cuda" if torch.cuda.is_available() else "cpu"
model = UNet().to(device)

In [ ]:
# ===============================
# 7️⃣ Loss + Optimizer
# ===============================
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
# ===============================
# 8️⃣ Training Loop
# ===============================
epochs = 10

for epoch in range(epochs):
    model.train()
    train_loss = 0
    for imgs, masks in tqdm(train_loader):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {train_loss/len(train_loader):.4f}")

In [ ]:
torch.save(trainer.model.state_dict(), "m2f_final_model.pth")

In [ ]:
# ===============================
# 9️⃣ Hybrid Prediction + Watershed
# ===============================
def hybrid_predict(model, image_path):
    model.eval()
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    input_tensor = torch.tensor(img_rgb.transpose(2,0,1), dtype=torch.float32).unsqueeze(0)/255.0
    input_tensor = input_tensor.to(device)
    with torch.no_grad():
        mask_pred = model(input_tensor)[0,0].cpu().numpy()
    
    combined_mask = (mask_pred > 0.5).astype(np.uint8)

    # Distance Transform
    distance = ndi.distance_transform_edt(combined_mask)
    coords = peak_local_max(distance, min_distance=15, labels=combined_mask)
    mask = np.zeros(distance.shape, dtype=bool)
    mask[tuple(coords.T)] = True
    markers, _ = ndi.label(mask)

    # Watershed
    final_labels = watershed(-distance, markers, mask=combined_mask)
    count = len(np.unique(final_labels)) - 1
    print(f"✅ Segmented {count} particles.")
    return final_labels

In [ ]:
# Core PyTorch + CUDA (already installed)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Computer vision
import cv2
import numpy as np
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from scipy import ndimage as ndi

# Dataset handling & augmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2
import json
import os
from tqdm import tqdm

# Visualization
import matplotlib.pyplot as plt

In [ ]:
class ParticleDataset(Dataset):
    def __init__(self, image_dir, json_file):
        with open(json_file) as f:
            data = json.load(f)
        self.images = data['images']
        self.annotations = data['annotations']
        self.image_dir = image_dir

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_path = os.path.join(self.image_dir, img_info['file_name'])
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = torch.tensor(img.transpose(2,0,1), dtype=torch.float32)/255.0

        mask = np.zeros((img_info['height'], img_info['width']), dtype=np.uint8)
        anns = [a for a in self.annotations if a['image_id'] == img_info['id']]
        for a in anns:
            x, y, w, h = a['bbox']
            mask[int(y):int(y+h), int(x):int(x+w)] = 1

        mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)
        return img, mask

In [ ]:
train_dataset = ParticleDataset(
    r"E:\Prithu\Sustain\sustain train\m2f_hybrid_dataset\train\images",
    r"E:\Prithu\Sustain\sustain train\m2f_hybrid_dataset\train.json"
)

val_dataset = ParticleDataset(
    r"E:\Prithu\Sustain\sustain train\m2f_hybrid_dataset\val\images",
    r"E:\Prithu\Sustain\sustain train\m2f_hybrid_dataset\val.json"
)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=2)

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class UNetPlusPlus(nn.Module):
    def __init__(self, in_ch=3, out_ch=1):
        super().__init__()
        filters = [64,128,256,512]
        self.enc1 = ConvBlock(in_ch, filters[0])
        self.enc2 = ConvBlock(filters[0], filters[1])
        self.enc3 = ConvBlock(filters[1], filters[2])
        self.enc4 = ConvBlock(filters[2], filters[3])
        self.pool = nn.MaxPool2d(2)

        self.up3 = nn.ConvTranspose2d(filters[3], filters[2], 2, stride=2)
        self.dec3 = ConvBlock(filters[3], filters[2])
        self.up2 = nn.ConvTranspose2d(filters[2], filters[1], 2, stride=2)
        self.dec2 = ConvBlock(filters[2], filters[1])
        self.up1 = nn.ConvTranspose2d(filters[1], filters[0], 2, stride=2)
        self.dec1 = ConvBlock(filters[1], filters[0])
        self.final = nn.Conv2d(filters[0], out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        d3 = self.up3(e4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return torch.sigmoid(self.final(d1))

model = UNetPlusPlus().to(device)

In [ ]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=15, factor=0.5, verbose=True)

best_val_loss = float("inf")
patience = 50
counter = 0
num_epochs = 500

In [ ]:
for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    for imgs, masks in tqdm(train_loader):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, masks)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            outputs = model(imgs)
            val_loss += criterion(outputs, masks).item()
    val_loss /= len(val_loader)
    scheduler.step(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pth")
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print("Early stopping triggered")
            break

In [ ]:
torch.save(model.state_dict(), "unet_particle_segmentation_best.pth")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = UNetPlusPlus().to(device)

model.load_state_dict(torch.load("unet_particle_segmentation_best.pth", map_location=device))

model.eval()

print("✅ Model loaded successfully")

In [ ]:
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"


def sliding_window_prediction(image_path, tile_size=1024, overlap=256):

    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    H, W = img_rgb.shape[:2]

    stride = tile_size - overlap

    full_mask = np.zeros((H, W))
    count_mask = np.zeros((H, W))

    for y in range(0, H - tile_size + 1, stride):
        for x in range(0, W - tile_size + 1, stride):

            tile = img_rgb[y:y+tile_size, x:x+tile_size]

            tensor = torch.tensor(tile.transpose(2,0,1), dtype=torch.float32)/255
            tensor = tensor.unsqueeze(0).to(device)

            with torch.no_grad():
                pred = model(tensor)

            pred_mask = pred[0,0].cpu().numpy()

            full_mask[y:y+tile_size, x:x+tile_size] += pred_mask
            count_mask[y:y+tile_size, x:x+tile_size] += 1

    full_mask = full_mask / np.maximum(count_mask,1)

    binary_mask = (full_mask > 0.5).astype(np.uint8)

    return img_rgb, binary_mask

In [ ]:
import torch
import cv2
import numpy as np
from tqdm import tqdm

def sliding_window_prediction(model, image_path, tile_size=1024, overlap=512, threshold=0.01, device="cuda"):
    """
    Run sliding window prediction on large image with overlap and merge tiles.

    Args:
        model: trained PyTorch segmentation model
        image_path: str, path to image
        tile_size: int, size of each tile
        overlap: int, overlapping pixels between tiles
        threshold: float, probability threshold for mask
        device: "cuda" or "cpu"

    Returns:
        full_img: original image as numpy array
        full_mask: predicted mask, same size as image
    """
    
    # Read image
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Image not found: {image_path}")
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    H, W, _ = img_rgb.shape

    # Prepare empty mask and counter for averaging overlapping tiles
    mask_accum = np.zeros((H, W), dtype=np.float32)
    count_accum = np.zeros((H, W), dtype=np.float32)

    stride = tile_size - overlap

    model.eval()
    with torch.no_grad():
        for y in tqdm(range(0, H, stride), desc="Sliding Window"):
            for x in range(0, W, stride):
                # Crop tile (make sure we don't go out of bounds)
                y1, y2 = y, min(y+tile_size, H)
                x1, x2 = x, min(x+tile_size, W)
                tile = img_rgb[y1:y2, x1:x2]

                # Pad if needed
                pad_y, pad_x = tile_size - (y2-y1), tile_size - (x2-x1)
                if pad_y > 0 or pad_x > 0:
                    tile = cv2.copyMakeBorder(tile, 0, pad_y, 0, pad_x, cv2.BORDER_REFLECT)

                # Preprocess tile
                tile_tensor = torch.tensor(tile.transpose(2,0,1), dtype=torch.float32).unsqueeze(0)/255.0
                tile_tensor = tile_tensor.to(device)

                # Predict
                pred = model(tile_tensor)[0,0].cpu().numpy()
                pred = pred[:y2-y1, :x2-x1]  # remove padding

                # Accumulate predictions
                mask_accum[y1:y2, x1:x2] += pred
                count_accum[y1:y2, x1:x2] += 1

    # Average overlapping tiles
    full_mask = mask_accum / np.maximum(count_accum, 1)
    
    # Apply threshold
    full_mask = (full_mask > threshold).astype(np.uint8)

    return img_rgb, full_mask

device = "cuda" if torch.cuda.is_available() else "cpu"
model = UNetPlusPlus().to(device)
model.load_state_dict(torch.load("unet_particle_segmentation_best.pth", map_location=device))

image_path = r"E:\Prithu\Sustain\sustain train\images\SUS24_07_Kathode_THI012_1000x_HF_07.jpg"
img, mask = sliding_window_prediction(model, image_path, tile_size=1024, overlap=512, threshold=0.1, device=device)

from scipy import ndimage as ndi
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from skimage.segmentation import find_boundaries
import matplotlib.pyplot as plt

# Particle separation using watershed
distance = ndi.distance_transform_edt(mask)
coords = peak_local_max(distance, min_distance=10, labels=mask)
marker_mask = np.zeros(distance.shape, dtype=bool)
marker_mask[tuple(coords.T)] = True
markers, _ = ndi.label(marker_mask)
labels = watershed(-distance, markers, mask=mask)

print("Particles detected:", len(np.unique(labels)) - 1)

# Overlay boundaries
boundaries = find_boundaries(labels)
overlay = img.copy()
overlay[boundaries] = [255,0,0]

plt.figure(figsize=(18,6))
plt.subplot(1,3,1)
plt.title("Original")
plt.imshow(img)
plt.subplot(1,3,2)
plt.title("Segmentation Mask")
plt.imshow(mask, cmap="gray")
plt.subplot(1,3,3)
plt.title("Particle Boundaries")
plt.imshow(overlay)
plt.show()

In [ ]:
image_path = r"E:\Prithu\Sustain\sustain train\images\SUS24_07_Kathode_THI012_1000x_HF_07.jpg"
import cv2
import matplotlib.pyplot as plt

img = cv2.imread(image_path)
if img is None:
    print("❌ Failed to load image. Check the path and file extension.")
else:
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.imshow(img_rgb)
    plt.title("Loaded Image")
    plt.show()
tile = img_rgb[:1024, :1024]
tile_tensor = torch.tensor(tile.transpose(2,0,1), dtype=torch.float32).unsqueeze(0)/255.0
tile_tensor = tile_tensor.to(device)

with torch.no_grad():
    pred = model(tile_tensor)[0,0].cpu().numpy()

plt.imshow(pred, cmap="hot")
plt.colorbar()
plt.title("Model Raw Prediction")
plt.show()

In [ ]:
import torch
import torch.nn as nn
import cv2
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from skimage.segmentation import watershed, find_boundaries
from skimage.feature import peak_local_max

# ======= 1. Model Definition (UNet++) =======

class ConvBlock(nn.Module):
    """Standard double convolution block used in UNet architectures."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class UNetPlusPlus(nn.Module):
    """Simplified UNet++ for binary particle segmentation."""
    def __init__(self, in_ch=3, out_ch=1):
        super().__init__()
        filters = [64, 128, 256, 512]
        self.enc1 = ConvBlock(in_ch, filters[0])
        self.enc2 = ConvBlock(filters[0], filters[1])
        self.enc3 = ConvBlock(filters[1], filters[2])
        self.enc4 = ConvBlock(filters[2], filters[3])
        self.pool = nn.MaxPool2d(2)

        self.up3 = nn.ConvTranspose2d(filters[3], filters[2], 2, stride=2)
        self.dec3 = ConvBlock(filters[3], filters[2])
        self.up2 = nn.ConvTranspose2d(filters[2], filters[1], 2, stride=2)
        self.dec2 = ConvBlock(filters[2], filters[1])
        self.up1 = nn.ConvTranspose2d(filters[1], filters[0], 2, stride=2)
        self.dec1 = ConvBlock(filters[1], filters[0])
        self.final = nn.Conv2d(filters[0], out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        d3 = self.up3(e4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        return torch.sigmoid(self.final(d1))

# ======= 2. Core Processing Functions =======

def sliding_window_prediction(model, image_path, device, tile_size=1024, overlap=256, threshold=0.05):
    """
    Processes large images by breaking them into overlapping tiles, 
    running prediction, and stitching them back together.
    """
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Image not found: {image_path}")
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    H, W = img_rgb.shape[:2]
    stride = tile_size - overlap
    
    mask_pred_full = np.zeros((H, W), dtype=np.float32)
    count_map = np.zeros((H, W), dtype=np.float32)

    for y in range(0, H, stride):
        for x in range(0, W, stride):
            # Calculate coordinates
            y1, x1 = y, x
            y2, x2 = min(y + tile_size, H), min(x + tile_size, W)
            
            # Extract tile and handle padding if tile is smaller than expected
            tile = img_rgb[y1:y2, x1:x2]
            
            # Prepare tensor
            tile_tensor = torch.tensor(tile.transpose(2,0,1), dtype=torch.float32).unsqueeze(0)/255.0
            tile_tensor = tile_tensor.to(device)
            
            with torch.no_grad():
                pred = model(tile_tensor)[0, 0].cpu().numpy()
            
            # Resize prediction to match current tile dimensions (handles edges)
            if pred.shape != (y2-y1, x2-x1):
                pred = cv2.resize(pred, (x2-x1, y2-y1))
                
            mask_pred_full[y1:y2, x1:x2] += pred
            count_map[y1:y2, x1:x2] += 1.0

    # Average overlaps and apply threshold
    mask_pred_full /= np.maximum(count_map, 1e-5)
    combined_mask = (mask_pred_full > threshold).astype(np.uint8)
    
    return img_rgb, combined_mask

def split_particles(mask, min_distance=10):
    """Separates touching particles using the Watershed algorithm."""
    # Compute Euclidean Distance Transform
    distance = ndi.distance_transform_edt(mask)
    
    # Find local maxima to act as markers
    coords = peak_local_max(distance, min_distance=min_distance, labels=mask)
    
    marker_mask = np.zeros(distance.shape, dtype=bool)
    marker_mask[tuple(coords.T)] = True
    markers, _ = ndi.label(marker_mask)
    
    # Run Watershed
    labels = watershed(-distance, markers, mask=mask)
    return labels

# ======= 3. Execution Block =======

if __name__ == "__main__":
    # Settings
    IMAGE_PATH = r"E:\Prithu\Sustain\sustain train\images\SUS24_07_Kathode_THI012_1000x_HF_07.jpg"
    MODEL_PATH = model.load_state_dict(torch.load("unet_particle_segmentation_best.pth", map_location=device))
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    # Initialize and Load Model
    model = UNetPlusPlus().to(DEVICE)
    try:
        model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
        model.eval()
        print("Model loaded successfully.")
    except FileNotFoundError:
        print(f"Warning: {MODEL_PATH} not found. Running with random weights for testing.")

    # Process Image
    img, mask = sliding_window_prediction(model, IMAGE_PATH, DEVICE, threshold=0.05)
    labels = split_particles(mask)

    # Visualization
    boundaries = find_boundaries(labels)
    overlay = img.copy()
    overlay[boundaries] = [255, 0, 0]  # Red boundaries

    plt.figure(figsize=(18, 6))
    plt.subplot(1, 3, 1)
    plt.title("Original Image")
    plt.imshow(img)
    plt.subplot(1, 3, 2)
    plt.title("Segmentation Mask (Threshold 0.05)")
    plt.imshow(mask, cmap="gray")
    plt.subplot(1, 3, 3)
    plt.title(f"Instance Detection ({len(np.unique(labels))-1} particles)")
    plt.imshow(overlay)
    plt.tight_layout()
    plt.show()

In [ ]:
# ===============================
# 8️⃣ Advanced Training Loop with Checkpoints & Early Stopping
# ===============================
import copy
from torch.optim.lr_scheduler import ReduceLROnPlateau
import matplotlib.pyplot as plt

# Optimizer, criterion, scheduler
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.BCELoss()  # Since UNet outputs sigmoid
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10, verbose=True)

# Early stopping and checkpointing
best_val_loss = float('inf')
early_stop_patience = 30
trigger_times = 0

num_epochs = 500
gradient_accumulation_steps = 4  # Helps with small batch sizes

for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    optimizer.zero_grad()
    
    for step, (imgs, masks) in enumerate(train_loader):
        imgs, masks = imgs.to(device), masks.to(device)
        outputs = model(imgs)
        loss = criterion(outputs, masks)
        loss = loss / gradient_accumulation_steps
        loss.backward()
        
        if (step + 1) % gradient_accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()
        
        train_loss += loss.item() * gradient_accumulation_steps
    
    train_loss /= len(train_loader)
    
    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, masks)
            val_loss += loss.item()
    val_loss /= len(val_loader)
    
    # Scheduler step
    scheduler.step(val_loss)
    
    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")
    
    # Checkpoint saving
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(best_model_wts, "best_unet_model.pth")
        print("✅ Saved Best Model")
        trigger_times = 0
    else:
        trigger_times += 1
        print(f"EarlyStopping counter: {trigger_times}/{early_stop_patience}")
        if trigger_times >= early_stop_patience:
            print("⚠️ Early stopping triggered")
            break
    
    # Optional: show a prediction from validation set
    if (epoch+1) % 10 == 0:
        sample_img, _ = next(iter(val_loader))
        sample_img = sample_img.to(device)
        with torch.no_grad():
            sample_pred = model(sample_img)[0,0].cpu().numpy()
        plt.figure(figsize=(6,6))
        plt.imshow(sample_pred, cmap='gray')
        plt.title(f"Sample Prediction at Epoch {epoch+1}")
        plt.show()

# Load the best model weights
model.load_state_dict(best_model_wts)
print("✅ Best model loaded for inference.")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
import os
import json
from tqdm import tqdm
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed, find_boundaries
import matplotlib.pyplot as plt

# ======================
# 1️⃣ Dataset
# ======================
class ParticleDataset(Dataset):
    def __init__(self, image_dir, json_file):
        with open(json_file) as f:
            data = json.load(f)
        self.image_dir = image_dir
        self.images = data['images']
        self.annotations = {}
        for ann in data['annotations']:
            img_id = ann['image_id']
            if img_id not in self.annotations:
                self.annotations[img_id] = []
            self.annotations[img_id].append(ann['bbox'])
        
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_path = os.path.join(self.image_dir, img_info['file_name'])
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        H, W, _ = img.shape
        
        # Create mask from bounding boxes
        mask = np.zeros((H,W), dtype=np.uint8)
        for bbox in self.annotations[img_info['id']]:
            x, y, w, h = map(int, bbox)
            mask[y:y+h, x:x+w] = 1
        
        img_tensor = torch.tensor(img.transpose(2,0,1), dtype=torch.float32)/255.0
        mask_tensor = torch.tensor(mask[None,...], dtype=torch.float32)
        return img_tensor, mask_tensor

# ======================
# 2️⃣ Model (Residual Attention UNet)
# ======================
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch)
        )
        self.relu = nn.ReLU(inplace=True)
        self.res_conv = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x):
        return self.relu(self.conv(x) + self.res_conv(x))

class AttentionBlock(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, 1), nn.BatchNorm2d(F_int))
        self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, 1), nn.BatchNorm2d(F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, 1), nn.Sigmoid())
    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.psi(torch.relu(g1 + x1))
        return x * psi

class ResAttentionUNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1):
        super().__init__()
        self.enc1 = ResidualBlock(in_ch, 64)
        self.enc2 = ResidualBlock(64, 128)
        self.enc3 = ResidualBlock(128, 256)
        self.pool = nn.MaxPool2d(2)
        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.att2 = AttentionBlock(128, 128, 64)
        self.dec2 = ResidualBlock(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.att1 = AttentionBlock(64, 64, 32)
        self.dec1 = ResidualBlock(128, 64)
        self.final = nn.Conv2d(64, out_ch, 1)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        d2 = self.up2(e3)
        e2_att = self.att2(d2, e2)
        d2 = self.dec2(torch.cat([d2, e2_att], dim=1))
        d1 = self.up1(d2)
        e1_att = self.att1(d1, e1)
        d1 = self.dec1(torch.cat([d1, e1_att], dim=1))
        return torch.sigmoid(self.final(d1))

# ======================
# 3️⃣ Loss Function
# ======================
def dice_loss(pred, target, smooth=1.):
    pred = pred.contiguous()
    target = target.contiguous()
    intersection = (pred * target).sum(dim=(2,3))
    loss = 1 - (2. * intersection + smooth) / (pred.sum(dim=(2,3)) + target.sum(dim=(2,3)) + smooth)
    return loss.mean()

criterion = lambda pred, target: F.binary_cross_entropy(pred, target) + dice_loss(pred, target)

# ======================
# 4️⃣ Training Setup
# ======================
device = "cuda" if torch.cuda.is_available() else "cpu"
train_dataset = ParticleDataset(
    r"E:\Prithu\Sustain\sustain train\m2f_hybrid_dataset\train\images",
    r"E:\Prithu\Sustain\sustain train\m2f_hybrid_dataset\train.json"
)

val_dataset = ParticleDataset(
    r"E:\Prithu\Sustain\sustain train\m2f_hybrid_dataset\val\images",
    r"E:\Prithu\Sustain\sustain train\m2f_hybrid_dataset\val.json"
)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=2)

model = ResAttentionUNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=10, verbose=True)

num_epochs = 500
patience = 25
best_val_loss = np.inf
counter = 0
save_path = "resatt_unet_particles.pth"
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=2)
# ======================
# 5️⃣ Sliding Window Prediction
# ======================
def sliding_window_predict(img, model, tile_size=1024, overlap=256, threshold=0.4):
    H, W, C = img.shape
    stride = tile_size - overlap
    mask_full = np.zeros((H,W), dtype=np.float32)
    for y in range(0, H, stride):
        for x in range(0, W, stride):
            y1, x1 = min(y+tile_size, H), min(x+tile_size, W)
            tile = img[y:y1, x:x1]
            tile_tensor = torch.tensor(tile.transpose(2,0,1), dtype=torch.float32).unsqueeze(0)/255.0
            tile_tensor = tile_tensor.to(device)
            with torch.no_grad():
                pred_tile = model(tile_tensor)[0,0].cpu().numpy()
            pred_tile_resized = cv2.resize(pred_tile, (x1-x, y1-y))
            mask_full[y:y1, x:x1] = np.maximum(mask_full[y:y1, x:x1], pred_tile_resized)
    binary_mask = (mask_full > threshold).astype(np.uint8)
    return binary_mask

# ======================
# 6️⃣ Particle Splitting
# ======================
def split_particles(mask, min_distance=15):
    distance = ndi.distance_transform_edt(mask)
    coords = peak_local_max(distance, min_distance=min_distance, labels=mask)
    marker_mask = np.zeros_like(distance, dtype=bool)
    if len(coords)>0:
        marker_mask[tuple(coords.T)] = True
        markers, _ = ndi.label(marker_mask)
    else:
        markers = np.zeros_like(distance)
    labels = watershed(-distance, markers, mask=mask)
    return labels

# ======================
# 7️⃣ Metric Function
# ======================
def compute_metrics(pred_mask, gt_mask):
    intersection = np.logical_and(pred_mask, gt_mask).sum()
    union = np.logical_or(pred_mask, gt_mask).sum()
    iou = intersection / union if union>0 else 0
    dice = (2*intersection)/(pred_mask.sum() + gt_mask.sum() + 1e-6)
    accuracy = (pred_mask == gt_mask).sum() / pred_mask.size
    return iou, dice, accuracy
class ParticleDataset(Dataset):
    def __init__(self, image_dir, json_file):
        with open(json_file) as f:
            data = json.load(f)
        self.image_dir = image_dir
        self.images = data['images']
        self.annotations = {}
        for ann in data['annotations']:
            img_id = ann['image_id']
            if img_id not in self.annotations:
                self.annotations[img_id] = []
            self.annotations[img_id].append(ann['bbox'])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_info = self.images[idx]
        img_path = os.path.join(self.image_dir, img_info['file_name'])
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        H, W, _ = img.shape

        # ✅ Safe access for images with no annotations
        mask = np.zeros((H, W), dtype=np.uint8)
        for bbox in self.annotations.get(img_info['id'], []):
            x, y, w, h = map(int, bbox)
            mask[y:y+h, x:x+w] = 1

        img_tensor = torch.tensor(img.transpose(2,0,1), dtype=torch.float32)/255.0
        mask_tensor = torch.tensor(mask[None,...], dtype=torch.float32)
        return img_tensor, mask_tensor
    # ======================
# 8️⃣ Training Loop with Checkpointing + Early Stopping + Validation Metrics
# ======================
for epoch in range(1, num_epochs+1):
    model.train()
    train_loss = 0
    for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs} Training"):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, masks)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)
    
    # Validation
    model.eval()
    val_loss = 0
    ious, dices, accs = [], [], []
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            outputs = model(imgs)
            val_loss += criterion(outputs, masks).item()
            # Metrics
            pred_mask = (outputs>0.5).cpu().numpy()
            gt_mask = masks.cpu().numpy()
            for pm, gm in zip(pred_mask, gt_mask):
                iou, dice, acc = compute_metrics(pm[0], gm[0])
                ious.append(iou)
                dices.append(dice)
                accs.append(acc)
    val_loss /= len(val_loader)
    scheduler.step(val_loss)
    
    print(f"Epoch {epoch}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | IoU: {np.mean(ious):.4f} | Dice: {np.mean(dices):.4f} | Acc: {np.mean(accs):.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), save_path)
        print(f"✅ Saved best model at Epoch {epoch}")
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print(f"⚠️ Early stopping at Epoch {epoch}")
            break

# ======================
# 9️⃣ Example Prediction Visualization
# ======================
val_img_path = r"E:\Prithu\Sustain\sustain train\images\SUS24_07_Kathode_THI012_1000x_HF_07.jpg"
img = cv2.imread(val_img_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
pred_mask = sliding_window_predict(img_rgb, model)
pred_labels = split_particles(pred_mask)
boundaries = find_boundaries(pred_labels)

overlay = img_rgb.copy()
overlay[boundaries] = [255,0,0]

plt.figure(figsize=(16,5))
plt.subplot(1,3,1); plt.title("Original"); plt.imshow(img_rgb)
plt.subplot(1,3,2); plt.title("Pred Mask"); plt.imshow(pred_mask, cmap="gray")
plt.subplot(1,3,3); plt.title("Pred Boundaries"); plt.imshow(overlay)
plt.show()

In [ ]:
import os, json, cv2, torch, random
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm
from scipy import ndimage as ndi
from sklearn.metrics import jaccard_score

# ==========================================
# 1. SETUP & GPU CONFIG
# ==========================================
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

BASE_DIR = r"E:\Prithu\Sustain\sustain train"
IMAGE_DIR = os.path.join(BASE_DIR, "images")
JSON_PATH = os.path.join(BASE_DIR, "sustain_fixed.json")
MODEL_SAVE_PATH = os.path.join(BASE_DIR, "best_hybrid_watershed_unet.pth")

print(f"📡 Device: {DEVICE}")
print(f"📁 Base Dir: {BASE_DIR}")
print(f"🖼 Images: {IMAGE_DIR}")
print(f"📄 JSON: {JSON_PATH}")
print(f"💾 Model will be saved at: {MODEL_SAVE_PATH}")

PATCH_SIZE = 512
OVERLAP = 128
STRIDE = PATCH_SIZE - OVERLAP
EPOCHS = 200
BATCH_SIZE = 4
PATIENCE = 25

# ==========================================
# 2. HYBRID ARCHITECTURE
# ==========================================
class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_c)
        )
        self.shortcut = nn.Sequential(nn.Conv2d(in_c, out_c, kernel_size=1), nn.BatchNorm2d(out_c))
    def forward(self, x): 
        return F.relu(self.conv(x) + self.shortcut(x))

class HybridWatershedUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = ResidualBlock(3, 64)
        self.enc2 = ResidualBlock(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ResidualBlock(128, 256)
        self.up1 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec1 = ResidualBlock(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec2 = ResidualBlock(128, 64)
        self.mask_out = nn.Conv2d(64, 2, kernel_size=1)   # Semantic masks
        self.dist_out = nn.Conv2d(64, 1, kernel_size=1)   # Distance map

    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(self.pool(s1))
        b = self.bottleneck(self.pool(s2))
        d1 = self.dec1(torch.cat([self.up1(b), s2], 1))
        d2 = self.dec2(torch.cat([self.up2(d1), s1], 1))
        return self.mask_out(d2), torch.sigmoid(self.dist_out(d2))

# ==========================================
# 3. LOSS FUNCTIONS
# ==========================================
class DiceBCELoss(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, inputs, targets, smooth=1):
        inputs = torch.sigmoid(inputs)
        inputs = inputs.view(-1)
        targets = targets.view(-1)
        intersection = (inputs * targets).sum()
        dice_loss = 1 - (2.*intersection + smooth)/(inputs.sum() + targets.sum() + smooth)
        BCE = F.binary_cross_entropy(inputs, targets)
        return BCE + dice_loss

# ==========================================
# 4. DATASET
# ==========================================
class TiledDataset(Dataset):
    def __init__(self, ids, augment=True):
        with open(JSON_PATH) as f: self.data = json.load(f)
        self.ids, self.augment = ids, augment
        self.cat_map = {1: 0, 2: 1} # 1:nRich, 2:Normal
        self.tiles = []

        for img_info in [i for i in self.data['images'] if i['id'] in ids]:
            for y in range(0, img_info['height'] - PATCH_SIZE + 1, STRIDE):
                for x in range(0, img_info['width'] - PATCH_SIZE + 1, STRIDE):
                    self.tiles.append({'img': img_info, 'x': x, 'y': y})

        # Augmentation pipeline
        self.aug = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.Transpose(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.2, rotate_limit=45, p=0.7),
            A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
            A.CLAHE(clip_limit=4.0, p=0.5),
            A.HueSaturationValue(p=0.3),
            A.GaussNoise(var_limit=(10,50), p=0.4)
        ])

    def __len__(self): return len(self.tiles)

    def __getitem__(self, idx):
        t = self.tiles[idx]
        img_path = os.path.join(IMAGE_DIR, t['img']['file_name'])
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        crop = img[t['y']:t['y']+PATCH_SIZE, t['x']:t['x']+PATCH_SIZE].astype(np.float32)/255.0

        # Masks
        mask = np.zeros((2, PATCH_SIZE, PATCH_SIZE), dtype=np.float32)
        for ann in [a for a in self.data['annotations'] if a['image_id']==t['img']['id']]:
            cid = self.cat_map.get(ann['category_id'], 1)
            for seg in ann['segmentation']:
                poly = (np.array(seg).reshape(-1,2) - [t['x'], t['y']]).astype(np.int32)
                if np.any((poly >=0) & (poly < PATCH_SIZE)):
                    cv2.fillPoly(mask[cid], [poly], 1.0)

        # Distance map
        full_binary = (mask[0]+mask[1]>0).astype(np.uint8)
        dist_map = ndi.distance_transform_edt(full_binary)
        if dist_map.max()>0: dist_map = dist_map / dist_map.max()

        # Augmentation
        if self.augment:
            aug_res = self.aug(image=crop, masks=[mask[0], mask[1], dist_map])
            img = torch.from_numpy(aug_res['image'].transpose(2,0,1)).float()
            mask = torch.stack([torch.from_numpy(m).float() for m in aug_res['masks'][:2]])
            dist_map = torch.from_numpy(aug_res['masks'][2]).unsqueeze(0).float()
        else:
            img = torch.from_numpy(crop.transpose(2,0,1)).float()
            mask = torch.from_numpy(mask).float()
            dist_map = torch.from_numpy(dist_map).unsqueeze(0).float()

        return img, mask, dist_map

# ==========================================
# 5. TRAIN / VALIDATION SPLIT
# ==========================================
ids = list(range(1,18))
random.shuffle(ids)
train_ds = TiledDataset(ids[:14], augment=True)
val_ds = TiledDataset(ids[14:], augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

# ==========================================
# 6. MODEL, OPTIMIZER, LOSS
# ==========================================
model = HybridWatershedUNet().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
criterion_mask = DiceBCELoss()
criterion_dist = nn.MSELoss()

best_v_loss = float('inf')
trigger = 0

print(f"🚀 Training on {DEVICE} with Early Stopping (patience={PATIENCE})")

# ==========================================
# 7. TRAINING LOOP
# ==========================================
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for imgs, masks, dists in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        imgs, masks, dists = imgs.to(DEVICE), masks.to(DEVICE), dists.to(DEVICE)
        optimizer.zero_grad()
        p_mask, p_dist = model(imgs)
        loss = criterion_mask(p_mask, masks) + 2.0*criterion_dist(p_dist, dists)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for imgs, masks, dists in val_loader:
            imgs, masks, dists = imgs.to(DEVICE), masks.to(DEVICE), dists.to(DEVICE)
            p_mask, p_dist = model(imgs)
            val_loss += (criterion_mask(p_mask, masks) + 2.0*criterion_dist(p_dist, dists)).item()
    avg_v_loss = val_loss/len(val_loader)

    print(f"Epoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {avg_v_loss:.4f}")

    # Save Best & Early Stopping
    if avg_v_loss < best_v_loss:
        best_v_loss = avg_v_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f"✅ Model Saved: {MODEL_SAVE_PATH}")
        trigger = 0
    else:
        trigger +=1
        if trigger >= PATIENCE:
            print("🛑 Early stopping triggered.")
            break

In [ ]:
import os
import cv2
import torch
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt

# ================= SETTINGS =================
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

BASE_DIR = r"E:\Prithu\Sustain\sustain train"
MODEL_PATH = os.path.join(BASE_DIR, "best_hybrid_watershed_unet.pth")

IMAGE_PATH = r"E:\Prithu\Sustain\sustain train\images\SUS23_01_Kathode_Ref1_quer_1000x_HF_03.jpg"

PATCH_SIZE = 512
OVERLAP = 128
STRIDE = PATCH_SIZE - OVERLAP

THRESH_N_RICH = 0.5
THRESH_NORMAL = 0.5

print(f"📡 Using {DEVICE}")

# ================= LOAD MODEL =================
model = HybridWatershedUNet().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

print("✅ Model Loaded")

# ================= LOAD IMAGE =================
img = cv2.cvtColor(cv2.imread(IMAGE_PATH), cv2.COLOR_BGR2RGB)
H, W, _ = img.shape

# ================= PADDING (CRITICAL FIX) =================
pad_h = (STRIDE - (H - PATCH_SIZE) % STRIDE) % STRIDE
pad_w = (STRIDE - (W - PATCH_SIZE) % STRIDE) % STRIDE

img_padded = cv2.copyMakeBorder(img, 0, pad_h, 0, pad_w, cv2.BORDER_REFLECT)
Hp, Wp, _ = img_padded.shape

# ================= OUTPUT MAPS =================
prob_map = np.zeros((2, Hp, Wp), dtype=np.float32)
count_map = np.zeros((Hp, Wp), dtype=np.float32)

# ================= SLIDING WINDOW =================
for y in range(0, Hp - PATCH_SIZE + 1, STRIDE):
    for x in range(0, Wp - PATCH_SIZE + 1, STRIDE):

        tile = img_padded[y:y+PATCH_SIZE, x:x+PATCH_SIZE].astype(np.float32) / 255.0
        tile = torch.from_numpy(tile.transpose(2,0,1)).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            p_mask, _ = model(tile)
            p_mask = torch.sigmoid(p_mask)[0].cpu().numpy()  # (2,H,W)

        prob_map[:, y:y+PATCH_SIZE, x:x+PATCH_SIZE] += p_mask
        count_map[y:y+PATCH_SIZE, x:x+PATCH_SIZE] += 1

# ================= NORMALIZE =================
prob_map /= count_map

# REMOVE PADDING
prob_map = prob_map[:, :H, :W]

# ================= SPLIT CLASSES =================
nrich_map = prob_map[0]
normal_map = prob_map[1]

# ================= THRESHOLD =================
nrich_mask = (nrich_map > THRESH_N_RICH).astype(np.uint8)
normal_mask = (normal_map > THRESH_NORMAL).astype(np.uint8)

# ================= OVERLAY =================
overlay = img.copy()

# Red = N-rich
overlay[nrich_mask == 1] = [255, 0, 0]

# Green = Normal
overlay[normal_mask == 1] = [0, 255, 0]

# ================= VISUALIZATION =================
plt.figure(figsize=(20,6))

# 1️⃣ Original
plt.subplot(1,5,1)
plt.title("Original")
plt.imshow(img)
plt.axis("off")

# 2️⃣ N-rich heatmap
plt.subplot(1,5,2)
plt.title("N-rich Heatmap")
plt.imshow(nrich_map, cmap='jet')
plt.axis("off")

# 3️⃣ Normal heatmap
plt.subplot(1,5,3)
plt.title("Normal Heatmap")
plt.imshow(normal_map, cmap='jet')
plt.axis("off")

# 4️⃣ Masks
plt.subplot(1,5,4)
plt.title("Masks")
plt.imshow(nrich_mask + normal_mask, cmap='gray')
plt.axis("off")

# 5️⃣ Overlay
plt.subplot(1,5,5)
plt.title("Segmentation")
plt.imshow(overlay)
plt.axis("off")

plt.tight_layout()
plt.show()

# ================= SAVE =================
cv2.imwrite(os.path.join(BASE_DIR, "nrich_mask.png"), nrich_mask*255)
cv2.imwrite(os.path.join(BASE_DIR, "normal_mask.png"), normal_mask*255)
cv2.imwrite(os.path.join(BASE_DIR, "overlay.png"), cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

print("💾 Saved: N-rich mask, Normal mask, Overlay")

In [ ]:
import os
import cv2
import torch
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm

# ========================== SETTINGS ==========================
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
BASE_DIR = r"E:\Prithu\Sustain\sustain train"
MODEL_PATH = os.path.join(BASE_DIR, "best_hybrid_watershed_unet.pth")

IMAGE_PATH = r"E:\Prithu\Sustain\sustain train\images\SUS23_01_Kathode_Ref1_langs_1000x_HF_01.jpg"
PATCH_SIZE = 512
OVERLAP = 128
STRIDE = PATCH_SIZE - OVERLAP
THRESHOLD = 0.5

print(f"📡 Using Device: {DEVICE}")

# ========================== MODEL ==========================
import torch.nn as nn
import torch.nn.functional as F

class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c)
        )
        self.shortcut = nn.Sequential(nn.Conv2d(in_c, out_c, 1), nn.BatchNorm2d(out_c))
    def forward(self, x):
        return F.relu(self.conv(x) + self.shortcut(x))

class HybridWatershedUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = ResidualBlock(3, 64)
        self.enc2 = ResidualBlock(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ResidualBlock(128, 256)
        self.up1 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec1 = ResidualBlock(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec2 = ResidualBlock(128, 64)
        self.mask_out = nn.Conv2d(64, 2, 1)   # 2 classes: n-rich + normal
        self.dist_out = nn.Conv2d(64, 1, 1)
    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(self.pool(s1))
        b = self.bottleneck(self.pool(s2))
        d1 = self.dec1(torch.cat([self.up1(b), s2],1))
        d2 = self.dec2(torch.cat([self.up2(d1), s1],1))
        return self.mask_out(d2), torch.sigmoid(self.dist_out(d2))

# Load trained model
model = HybridWatershedUNet().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print("✅ Model Loaded")

# ========================== LOAD IMAGE ==========================
img = cv2.cvtColor(cv2.imread(IMAGE_PATH), cv2.COLOR_BGR2RGB)
H, W, _ = img.shape

# Prepare full-size heatmaps
nrich_map = np.zeros((H, W), dtype=np.float32)
normal_map = np.zeros((H, W), dtype=np.float32)

# ========================== SLIDING WINDOW PREDICTION WITH FULL COVERAGE ==========================
y_positions = list(range(0, H - PATCH_SIZE + 1, STRIDE))
x_positions = list(range(0, W - PATCH_SIZE + 1, STRIDE))

# Add last patch if needed to cover right/bottom edges
if y_positions[-1] != H - PATCH_SIZE:
    y_positions.append(H - PATCH_SIZE)
if x_positions[-1] != W - PATCH_SIZE:
    x_positions.append(W - PATCH_SIZE)

for y in tqdm(y_positions, desc="Sliding Window"):
    for x in x_positions:
        tile = img[y:y+PATCH_SIZE, x:x+PATCH_SIZE].astype(np.float32)/255.0
        tile_tensor = torch.from_numpy(tile.transpose(2,0,1)).unsqueeze(0).float().to(DEVICE)

        with torch.no_grad():
            pred_mask, _ = model(tile_tensor)
            pred_mask = torch.sigmoid(pred_mask).cpu().numpy()[0]  # (2,H,W)

        # Merge predictions using max (to handle overlaps)
        nrich_map[y:y+PATCH_SIZE, x:x+PATCH_SIZE] = np.maximum(
            nrich_map[y:y+PATCH_SIZE, x:x+PATCH_SIZE], pred_mask[0]
        )
        normal_map[y:y+PATCH_SIZE, x:x+PATCH_SIZE] = np.maximum(
            normal_map[y:y+PATCH_SIZE, x:x+PATCH_SIZE], pred_mask[1]
        )

# ========================== THRESHOLD ==========================
nrich_bin = (nrich_map > THRESHOLD).astype(np.uint8)
normal_bin = (normal_map > THRESHOLD).astype(np.uint8)

# ========================== OVERLAY ==========================
overlay = img.copy()
colored = np.zeros_like(img)
colored[:,:,0] = nrich_bin * 255   # Red channel for n-rich
colored[:,:,1] = normal_bin * 255  # Green channel for normal

alpha = 0.5
overlay = cv2.addWeighted(img, 1.0, colored, alpha, 0)

# ========================== VISUALIZATION ==========================
plt.figure(figsize=(20,5))

plt.subplot(1,5,1)
plt.title("Original")
plt.imshow(img)
plt.axis("off")

plt.subplot(1,5,2)
plt.title("N-rich Heatmap")
plt.imshow(nrich_map, cmap='Reds')
plt.axis("off")

plt.subplot(1,5,3)
plt.title("Normal Heatmap")
plt.imshow(normal_map, cmap='Greens')
plt.axis("off")

plt.subplot(1,5,4)
plt.title("Binary Masks")
combined_bin = np.zeros_like(img)
combined_bin[:,:,0] = nrich_bin*255
combined_bin[:,:,1] = normal_bin*255
plt.imshow(combined_bin)
plt.axis("off")

plt.subplot(1,5,5)
plt.title("Overlay Segmentation")
plt.imshow(overlay)
plt.axis("off")

plt.tight_layout()
plt.show()

# ========================== SAVE ==========================
cv2.imwrite(os.path.join(BASE_DIR, "nrich_mask.png"), nrich_bin*255)
cv2.imwrite(os.path.join(BASE_DIR, "normal_mask.png"), normal_bin*255)
cv2.imwrite(os.path.join(BASE_DIR, "overlay.png"), cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

print("💾 Prediction saved: masks + overlay")

In [ ]:
import os
import cv2
import torch
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm

# ========================== SETTINGS ==========================
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
BASE_DIR = r"E:\Prithu\Sustain\sustain train"
MODEL_PATH = os.path.join(BASE_DIR, "best_hybrid_watershed_unet.pth")

IMAGE_PATH = r"E:\Prithu\Sustain Image with COCO JSON\images\sus23_01_kathode_ref1_quer_1000x_hf_04.jpg"
PATCH_SIZE = 512
OVERLAP = 128
STRIDE = PATCH_SIZE - OVERLAP
THRESHOLD = 0.5

print(f"📡 Using Device: {DEVICE}")

# ========================== MODEL ==========================
import torch.nn as nn
import torch.nn.functional as F

class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1),
            nn.BatchNorm2d(out_c)
        )
        self.shortcut = nn.Sequential(nn.Conv2d(in_c, out_c, 1), nn.BatchNorm2d(out_c))
    def forward(self, x):
        return F.relu(self.conv(x) + self.shortcut(x))

class HybridWatershedUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = ResidualBlock(3, 64)
        self.enc2 = ResidualBlock(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ResidualBlock(128, 256)
        self.up1 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec1 = ResidualBlock(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec2 = ResidualBlock(128, 64)
        self.mask_out = nn.Conv2d(64, 2, 1)   # 2 classes: n-rich + normal
        self.dist_out = nn.Conv2d(64, 1, 1)
    def forward(self, x):
        s1 = self.enc1(x)
        s2 = self.enc2(self.pool(s1))
        b = self.bottleneck(self.pool(s2))
        d1 = self.dec1(torch.cat([self.up1(b), s2],1))
        d2 = self.dec2(torch.cat([self.up2(d1), s1],1))
        return self.mask_out(d2), torch.sigmoid(self.dist_out(d2))

# Load trained model
model = HybridWatershedUNet().to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print("✅ Model Loaded")

# ========================== LOAD IMAGE ==========================
img = cv2.cvtColor(cv2.imread(IMAGE_PATH), cv2.COLOR_BGR2RGB)
H, W, _ = img.shape

# Prepare full-size heatmaps
nrich_map = np.zeros((H, W), dtype=np.float32)
normal_map = np.zeros((H, W), dtype=np.float32)

# ========================== SLIDING WINDOW PREDICTION WITH FULL COVERAGE ==========================
y_positions = list(range(0, H - PATCH_SIZE + 1, STRIDE))
x_positions = list(range(0, W - PATCH_SIZE + 1, STRIDE))

# Add last patch if needed to cover right/bottom edges
if y_positions[-1] != H - PATCH_SIZE:
    y_positions.append(H - PATCH_SIZE)
if x_positions[-1] != W - PATCH_SIZE:
    x_positions.append(W - PATCH_SIZE)

for y in tqdm(y_positions, desc="Sliding Window"):
    for x in x_positions:
        tile = img[y:y+PATCH_SIZE, x:x+PATCH_SIZE].astype(np.float32)/255.0
        tile_tensor = torch.from_numpy(tile.transpose(2,0,1)).unsqueeze(0).float().to(DEVICE)

        with torch.no_grad():
            pred_mask, _ = model(tile_tensor)
            pred_mask = torch.sigmoid(pred_mask).cpu().numpy()[0]  # (2,H,W)

        # Merge predictions using max (to handle overlaps)
        nrich_map[y:y+PATCH_SIZE, x:x+PATCH_SIZE] = np.maximum(
            nrich_map[y:y+PATCH_SIZE, x:x+PATCH_SIZE], pred_mask[0]
        )
        normal_map[y:y+PATCH_SIZE, x:x+PATCH_SIZE] = np.maximum(
            normal_map[y:y+PATCH_SIZE, x:x+PATCH_SIZE], pred_mask[1]
        )

# ========================== THRESHOLD ==========================
nrich_bin = (nrich_map > THRESHOLD).astype(np.uint8)
normal_bin = (normal_map > THRESHOLD).astype(np.uint8)

# ========================== OVERLAY ==========================
overlay = img.copy()
colored = np.zeros_like(img)
colored[:,:,0] = nrich_bin * 255   # Red channel for n-rich
colored[:,:,1] = normal_bin * 255  # Green channel for normal

alpha = 0.5
overlay = cv2.addWeighted(img, 1.0, colored, alpha, 0)

# ========================== VISUALIZATION ==========================
plt.figure(figsize=(20,5))

plt.subplot(1,5,1)
plt.title("Original")
plt.imshow(img)
plt.axis("off")

plt.subplot(1,5,2)
plt.title("N-rich Heatmap")
plt.imshow(nrich_map, cmap='Reds')
plt.axis("off")

plt.subplot(1,5,3)
plt.title("Normal Heatmap")
plt.imshow(normal_map, cmap='Greens')
plt.axis("off")

plt.subplot(1,5,4)
plt.title("Binary Masks")
combined_bin = np.zeros_like(img)
combined_bin[:,:,0] = nrich_bin*255
combined_bin[:,:,1] = normal_bin*255
plt.imshow(combined_bin)
plt.axis("off")

plt.subplot(1,5,5)
plt.title("Overlay Segmentation")
plt.imshow(overlay)
plt.axis("off")

plt.tight_layout()
plt.show()

# ========================== SAVE ==========================
cv2.imwrite(os.path.join(BASE_DIR, "nrich_mask.png"), nrich_bin*255)
cv2.imwrite(os.path.join(BASE_DIR, "normal_mask.png"), normal_bin*255)
cv2.imwrite(os.path.join(BASE_DIR, "overlay.png"), cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))

print("💾 Prediction saved: masks + overlay")